# Interactive River Network Map — Colorado & Chattahoochee, Compressed ⇄ Uncompressed

Builds the site map (`index.html`) with two switches in the top-right corner:

- **River:** Colorado | Chattahoochee
- **View:** Compressed | Uncompressed

| River | Compressed graph | Uncompressed graph |
|---|---|---|
| Colorado | `master_compressed_river_graph.pkl` (67 nodes) | `largest_river_component.pkl` (67,401 pixels) |
| Chattahoochee | `chattahoochee_structural_2025.pkl` (160 nodes) | `chattahoochee_full_2025.pkl` (20,871 pixels) |

**Compressed view:** retained nodes, edges drawn along the real river path, water-quality stations
(one symbol for all of them, since models are evaluated with leave-one-out), hydrology support, sources, outlet, junctions.

**Uncompressed view:** every river-pixel node. They are drawn on a canvas only for the area on
screen once you zoom in (zoom ≥ 11), so the page stays fast. Click a pixel to see its raster row/col,
distance to the outlet, number of upstream pixels and which compressed edge it was folded into.

Link straight to a view with e.g. `index.html#chattahoochee/uncompressed`
(`#compressed` / `#uncompressed` alone mean Colorado).

To add another river, add an entry to `RIVERS` below.

### Output
- `figures/interactive_river_network_map_toggle.html`
- `index.html` (the published site)

In [ ]:
import os
import json
import math
import pickle
import html
from getpass import getpass

import numpy as np
import pandas as pd
import networkx as nx
import folium
from folium import plugins
from branca.element import Element, MacroElement
from jinja2 import Template

RIVERS = {
    "colorado": {
        "label": "Colorado",
        "compressed": "graphs/colorado/master_compressed_river_graph.pkl",
        "uncompressed": "graphs/colorado/largest_river_component.pkl",
    },
    "chattahoochee": {
        "label": "Chattahoochee",
        "compressed": "graphs/chattahoochee/chattahoochee_structural_2025.pkl",
        "uncompressed": "graphs/chattahoochee/chattahoochee_full_2025.pkl",
    },
}
DEFAULT_RIVER = "colorado"

OUTPUT_HTML = "figures/interactive_river_network_map_toggle.html"
SITE_HTML = "index.html"          # set to None to skip updating the site
OUTPUT_POINTS_CSV = "data/interactive_river_map_points_all_rivers.csv"

# Uncompressed pixel nodes are only drawn at/after this zoom level
PIXEL_MIN_ZOOM = 11
PIXEL_MAX_IN_VIEW = 8000

CORE_FEATURES = {"DO concentration", "Water temperature", "Specific conductance", "Turbidity", "pH"}

print("Working directory:", os.getcwd())
for key, cfg in RIVERS.items():
    for view in ["compressed", "uncompressed"]:
        print(f"{cfg['label']:14s} {view:13s} exists={os.path.exists(cfg[view])}  {cfg[view]}")

## 1. Load the graphs

In [ ]:
rivers = {}

for key, cfg in RIVERS.items():
    with open(cfg["compressed"], "rb") as f:
        Gc = pickle.load(f)
    with open(cfg["uncompressed"], "rb") as f:
        Gu = pickle.load(f)
    rivers[key] = {"key": key, "label": cfg["label"], "Gc": Gc, "Gu": Gu}

    print(cfg["label"])
    for name, graph in [("compressed", Gc), ("uncompressed", Gu)]:
        print(f"  {name:13s} {graph.number_of_nodes():>7,} nodes  {graph.number_of_edges():>7,} edges  "
              f"weak components = {nx.number_weakly_connected_components(graph)}")
    print("  compressed nodes missing from uncompressed graph:", sum(n not in Gu for n in Gc))

## 2. Analyse each river

- structural nodes (sources, outlets, junctions) in both graphs
- water-quality station nodes and hydrology support gauges
- distance from each source to the first downstream measurement
- per-pixel distance to outlet, upstream pixel count and compressed edge membership

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    r = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def structural_nodes(graph):
    sources, outlets, junctions = set(), set(), set()
    for node in graph:
        indeg, outdeg = graph.in_degree(node), graph.out_degree(node)
        if indeg == 0 and outdeg > 0:
            sources.add(node)
        if indeg > 0 and outdeg == 0:
            outlets.add(node)
        if indeg + outdeg >= 3:
            junctions.add(node)
    return sources, outlets, junctions


def wq_station_info(Gc, node):
    """(station ids, available features, extra popup details) for a compressed node.

    Colorado stores stations as `usgs_clusters`; Chattahoochee as `usgs_stations`
    plus a `wq_2025_summary` of the downloaded 2025 data.
    """
    d = Gc.nodes[node]
    if d.get("usgs_clusters"):
        clusters = d["usgs_clusters"]
        members = sorted({s for c in clusters for s in c.get("member_stations", [])})
        features = sorted({f for c in clusters for f in c.get("features", [])})
        return members, features, {}
    stations = d.get("usgs_stations") or []
    if stations:
        members = sorted(s["monitoring_location_id"] for s in stations)
        features = sorted({v for s in stations for v in s.get("variables", [])})
        extra = {"Names": "; ".join(s.get("station_name", "") for s in stations)}
        summary = d.get("wq_2025_summary") or {}
        obs = [
            f"{name}: {info['count']:,} obs (median {info['median']:g})"
            for name, info in sorted(summary.items())
            if info.get("count")
        ]
        if summary:
            extra["2025 data"] = "; ".join(obs) if obs else "none downloaded"
        return members, features, extra
    return [], [], {}


def analyse(r):
    Gc, Gu = r["Gc"], r["Gu"]

    r["c_sources"], r["c_outlets"], r["c_junctions"] = structural_nodes(Gc)
    r["u_sources"], r["u_outlets"], r["u_junctions"] = structural_nodes(Gu)

    r["wq_nodes"] = {n for n in Gc if wq_station_info(Gc, n)[0]}
    r["support_nodes"] = {n for n, d in Gc.nodes(data=True) if d.get("support_gauge_count", 0) > 0}
    # Stations snapped directly to pixels in main.ipynb (Colorado only)
    r["snapped_nodes"] = {n for n, d in Gu.nodes(data=True) if d.get("station_count", 0) > 0}

    # ---- source coverage on the compressed graph ----
    measurement_nodes = r["wq_nodes"] | r["support_nodes"] | {
        n for n, d in Gc.nodes(data=True) if d.get("usgs_station_count", 0) > 0
    }
    rows = []
    for source in r["c_sources"]:
        lengths = nx.single_source_dijkstra_path_length(Gc, source, weight="length_m")
        reachable = {n: dist for n, dist in lengths.items() if n in measurement_nodes}
        nearest = min(reachable, key=reachable.get) if reachable else None
        rows.append({
            "river": r["label"],
            "source_node": source,
            "nearest_measurement_node": nearest,
            "distance_to_first_measurement_km": reachable[nearest] / 1000 if reachable else np.nan,
        })
    r["source_coverage"] = pd.DataFrame(rows)
    r["source_distance"] = r["source_coverage"].set_index("source_node")[
        "distance_to_first_measurement_km"].to_dict()

    # ---- per-pixel attributes on the uncompressed graph ----
    def step_m(a, b):
        na, nb = Gu.nodes[a], Gu.nodes[b]
        return haversine_m(na["latitude"], na["longitude"], nb["latitude"], nb["longitude"])

    topo = list(nx.topological_sort(Gu))
    dist = {}
    for node in reversed(topo):
        succ = list(Gu.successors(node))
        dist[node] = 0.0 if not succ else min(dist[s] + step_m(node, s) for s in succ)
    upstream = {}
    for node in topo:
        upstream[node] = sum(upstream[p] + 1 for p in Gu.predecessors(node))
    r["dist_to_outlet_m"], r["upstream_count"] = dist, upstream

    segments, segment_of = [], {}     # segments: [u, v, pixel_edges, path_length_km]
    for k, (u, v, data) in enumerate(Gc.edges(data=True)):
        path = data.get("original_path", [u, v])
        length_m = sum(step_m(path[i], path[i + 1]) for i in range(len(path) - 1))
        segments.append([int(u), int(v), len(path) - 1, round(length_m / 1000, 2)])
        for node in path[1:-1]:
            segment_of[node] = k
    r["segments"], r["segment_of"] = segments, segment_of

    lats = [float(d["latitude"]) for _, d in Gu.nodes(data=True)]
    lons = [float(d["longitude"]) for _, d in Gu.nodes(data=True)]
    r["bounds"] = [[min(lats), min(lons)], [max(lats), max(lons)]]

    return {
        "River": r["label"],
        "Compressed nodes": Gc.number_of_nodes(),
        "Uncompressed nodes": Gu.number_of_nodes(),
        "Pixels accounted for": len(segment_of) + Gc.number_of_nodes(),
        "Sources": len(r["c_sources"]),
        "Junctions": len(r["c_junctions"]),
        "Outlets": len(r["c_outlets"]),
        "WQ station nodes": len(r["wq_nodes"]),
        "Hydrology support": len(r["support_nodes"]),
        "Snapped WQP stations": len(r["snapped_nodes"]),
        "River length (km)": round(sum(s[3] for s in segments)),
        "Longest path to outlet (km)": round(max(dist.values()) / 1000),
    }


summary = pd.DataFrame([analyse(r) for r in rivers.values()]).set_index("River").T
display(summary)

## 3. Build a searchable point table

In [ ]:
point_rows = []

for key, r in rivers.items():
    Gc, Gu = r["Gc"], r["Gu"]

    def add_point(node, point_type, label, extra=None):
        d = Gu.nodes[node]
        row = {"river": key, "node_id": node, "type": point_type, "label": label,
               "latitude": float(d["latitude"]), "longitude": float(d["longitude"])}
        row.update(extra or {})
        point_rows.append(row)

    for node in sorted(r["c_sources"]):
        add_point(node, "Network source", f"Source node {node}")
    for node in sorted(r["c_outlets"]):
        add_point(node, "Outlet / downstream leaf", f"Outlet node {node}")
    for node in sorted(r["c_junctions"]):
        add_point(node, "Junction / meeting point", f"Junction node {node}")
    for node in sorted(r["wq_nodes"]):
        members, features, _ = wq_station_info(Gc, node)
        add_point(node, "Water-quality station", " / ".join(members),
                  {"member_stations": ", ".join(members), "feature_count": len(features)})
    for node in sorted(r["support_nodes"]):
        gauges = Gc.nodes[node].get("support_gauges", [])
        ids = sorted({str(g["station_id"]) for g in gauges if g.get("station_id") is not None})
        add_point(node, "Hydrology support", " / ".join(ids) or f"node {node}",
                  {"support_gauges": ", ".join(ids)})
    for node in sorted(r["snapped_nodes"]):
        stations = Gu.nodes[node]["stations"]
        label = " / ".join(f"{s['station_id']} {s.get('station_name', '')}".strip() for s in stations)
        add_point(node, "WQ station (snapped to pixel)", label)

points_df = pd.DataFrame(point_rows)
points_df.to_csv(OUTPUT_POINTS_CSV, index=False)
print("Saved:", OUTPUT_POINTS_CSV)
print(points_df.groupby(["river", "type"]).size())

## 4. Create the interactive map with the river and compressed / uncompressed switches

In [ ]:
# ==================================================
# CARTO BASEMAP API KEY
# (read from the CARTO_API_KEY environment variable, otherwise prompt)
# ==================================================

CARTO_API_KEY = os.environ.get("CARTO_API_KEY", "").strip()
if not CARTO_API_KEY:
    CARTO_API_KEY = getpass("Enter your CARTO Basemaps API key: ").strip()
if not CARTO_API_KEY:
    raise ValueError("A CARTO Basemaps API key is required.")


# ==================================================
# MAP + BASEMAPS
# ==================================================

start_bounds = rivers[DEFAULT_RIVER]["bounds"]
m = folium.Map(
    location=[(start_bounds[0][0] + start_bounds[1][0]) / 2, (start_bounds[0][1] + start_bounds[1][1]) / 2],
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

OSM_ATTR = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
CARTO_ATTR = OSM_ATTR + ' &copy; <a href="https://carto.com/attribution/">CARTO</a>'

tile_osm = folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap", attr=OSM_ATTR, max_zoom=19, show=True,
).add_to(m)
tile_light = folium.TileLayer(
    tiles="https://basemaps.cartocdn.com/rastertiles/light_all/{z}/{x}/{y}{r}.png" f"?key={CARTO_API_KEY}",
    name="CARTO Positron", attr=CARTO_ATTR, max_zoom=20, show=False,
).add_to(m)
tile_dark = folium.TileLayer(
    tiles="https://basemaps.cartocdn.com/rastertiles/dark_all/{z}/{x}/{y}{r}.png" f"?key={CARTO_API_KEY}",
    name="CARTO Dark Matter", attr=CARTO_ATTR, max_zoom=20, show=False,
).add_to(m)


# ==================================================
# HELPERS
# ==================================================

def google_maps_url(lat, lon):
    return f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"


def make_helpers(r):
    """Popup / coordinate helpers bound to one river."""
    Gu = r["Gu"]

    def coords(node):
        d = Gu.nodes[node]
        return float(d["latitude"]), float(d["longitude"])

    def popup_html(node, title, details=None):
        lat, lon = coords(node)
        rows = [
            f"<b>{html.escape(str(title))}</b> <span style='color:#666'>({html.escape(r['label'])})</span>",
            f"<b>Graph node:</b> {node}",
            f"<b>Latitude:</b> {lat:.6f}",
            f"<b>Longitude:</b> {lon:.6f}",
        ]
        for key, value in (details or {}).items():
            if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
                continue
            rows.append(f"<b>{html.escape(str(key))}:</b> {html.escape(str(value))}")
        rows.append(f'<a href="{google_maps_url(lat, lon)}" target="_blank">Open in Google Maps</a>')
        return folium.Popup("<br>".join(rows), max_width=440)

    def pixel_details(node):
        d = Gu.nodes[node]
        out = {
            "Raster row / col": f"{d['row']} / {d['col']}",
            "Distance to outlet": f"{r['dist_to_outlet_m'][node] / 1000:,.1f} km",
            "Upstream pixels": f"{r['upstream_count'][node]:,}",
        }
        if d.get("upstream_area_km2") is not None:
            out["Upstream area"] = f"{d['upstream_area_km2']:,.0f} km²"
        if d.get("elevation_m") is not None:
            out["Elevation"] = f"{d['elevation_m']:,.1f} m"
        return out

    def path_coords(path):
        # 5 decimals ≈ 1 m, plenty for ~90 m MERIT pixels and keeps the HTML small
        return [[round(c, 5) for c in coords(n)] for n in path if n in Gu]

    return coords, popup_html, pixel_details, path_coords


def coverage(r, node):
    distance = r["source_distance"].get(node, np.nan)
    if pd.isna(distance):
        return "red", "No downstream measurement"
    color = "green" if distance <= 100 else "orange"
    return color, f"{distance:.1f} km to first downstream measurement"


# "<river>:<view>" -> list of (layer name, layer, visible by default)
VIEW_LAYERS = {}


def draw_river(r):
    key, Gc, Gu = r["key"], r["Gc"], r["Gu"]
    coords, popup_html, pixel_details, path_coords = make_helpers(r)
    views = {"compressed": [], "uncompressed": []}

    def group(name, view_list, show=True):
        layer = folium.FeatureGroup(name=f"{r['label']} — {name}", show=False).add_to(m)
        for v in view_list:
            views[v].append((name, layer, show))
        return layer

    C, U, BOTH = ["compressed"], ["uncompressed"], ["compressed", "uncompressed"]

    c_river = group("River network (compressed edges)", C)
    c_nodes = group(f"Compressed nodes ({Gc.number_of_nodes():,})", C)
    u_river = group("River network (pixel chains)", U)
    u_pixels = group(f"River-pixel nodes ({Gu.number_of_nodes():,}, zoom ≥ {PIXEL_MIN_ZOOM})", U)
    layers = {
        "compressed": [group(n, C) for n in
                       ["Network sources", "Outlets / downstream leaves", "Junctions / meeting points"]],
        "uncompressed": [group(n, U) for n in
                         ["Network sources", "Outlets / downstream leaves", "Junctions / meeting points"]],
    }
    u_snapped = group("WQ stations snapped to pixels", U) if r["snapped_nodes"] else None
    wq_layer = group("Water-quality stations", BOTH)
    support_layer = group("Hydrology support", BOTH) if r["support_nodes"] else None
    r["pixel_layer"] = u_pixels

    # ---- river geometry (same path in both views) ----
    for k, (u, v, data) in enumerate(Gc.edges(data=True)):
        line = path_coords(data.get("original_path", [u, v]))
        if len(line) < 2:
            line = [list(coords(u)), list(coords(v))]
        _, _, pixel_edges, length_km = r["segments"][k]
        folium.PolyLine(
            line, weight=3, opacity=0.7, color="#3388ff",
            tooltip=f"Compressed edge {u} → {v} | {data.get('length_m', 0) / 1000:.1f} km | "
                    f"replaces {pixel_edges:,} pixel edges",
        ).add_to(c_river)
        folium.PolyLine(
            line, weight=2, opacity=0.85, color="#1e40af",
            tooltip=f"Pixel chain {u} … {v} | {pixel_edges + 1:,} pixel nodes | {length_km:.1f} km",
        ).add_to(u_river)

    # ---- compressed nodes ----
    for node in Gc:
        members, _, _ = wq_station_info(Gc, node)
        folium.CircleMarker(
            list(coords(node)), radius=4, color="#1e3a8a", weight=1,
            fill=True, fill_color="#93c5fd", fill_opacity=0.95,
            tooltip=f"Compressed node {node}",
            popup=popup_html(node, "Compressed node", {
                "In-degree": Gc.in_degree(node),
                "Out-degree": Gc.out_degree(node),
                "WQ stations": ", ".join(members),
            }),
        ).add_to(c_nodes)

    # ---- structural nodes, one set per view ----
    for view, graph, extra in [("compressed", Gc, lambda n: {}), ("uncompressed", Gu, pixel_details)]:
        s_layer, o_layer, j_layer = layers[view]
        prefix = "c" if view == "compressed" else "u"
        for node in r[f"{prefix}_sources"]:
            color, coverage_text = coverage(r, node)
            folium.Marker(
                list(coords(node)), tooltip=f"Source {node}",
                popup=popup_html(node, "Network source", {
                    "Coverage": coverage_text,
                    "In-degree": graph.in_degree(node), "Out-degree": graph.out_degree(node),
                    **extra(node),
                }),
                icon=folium.Icon(color=color, icon="play", prefix="fa"),
            ).add_to(s_layer)
        for node in r[f"{prefix}_outlets"]:
            folium.Marker(
                list(coords(node)), tooltip=f"Outlet {node}",
                popup=popup_html(node, "Outlet / downstream leaf", {
                    "In-degree": graph.in_degree(node), "Out-degree": graph.out_degree(node),
                    **extra(node),
                }),
                icon=folium.Icon(color="darkgreen", icon="stop", prefix="fa"),
            ).add_to(o_layer)
        for node in r[f"{prefix}_junctions"]:
            folium.CircleMarker(
                list(coords(node)), radius=7, color="#f59e0b", weight=2,
                fill=True, fill_color="#f59e0b", fill_opacity=0.75,
                tooltip=f"Junction {node}",
                popup=popup_html(node, "Junction / meeting point", {
                    "In-degree": graph.in_degree(node), "Out-degree": graph.out_degree(node),
                    **extra(node),
                }),
            ).add_to(j_layer)

    # ---- water-quality stations (one symbol, shared by both views) ----
    for node in r["wq_nodes"]:
        members, features, extra = wq_station_info(Gc, node)
        details = {
            "Stations": ", ".join(members),
            **extra,
            "Core features": f"{len(set(features) & CORE_FEATURES)}/{len(CORE_FEATURES)}",
            "Available": ", ".join(features),
            "Missing": ", ".join(sorted(CORE_FEATURES - set(features))),
            **pixel_details(node),
        }
        folium.Marker(
            list(coords(node)),
            tooltip="WQ: " + " / ".join(members),
            popup=popup_html(node, "Water-quality station", details),
            icon=folium.Icon(color="blue", icon="flask", prefix="fa"),
        ).add_to(wq_layer)

    # ---- hydrology support ----
    for node in r["support_nodes"]:
        gauges = Gc.nodes[node].get("support_gauges", [])
        gauge_ids = sorted({str(g["station_id"]) for g in gauges if g.get("station_id") is not None})
        used_by = sorted({s for g in gauges for s in g.get("used_by_wq_stations", [])})
        folium.Marker(
            list(coords(node)),
            tooltip="HYDROLOGY: " + (" / ".join(gauge_ids) if gauge_ids else str(node)),
            popup=popup_html(node, "Hydrology support", {
                "Gauge IDs": ", ".join(gauge_ids),
                "Used by WQ stations": ", ".join(used_by),
                **pixel_details(node),
            }),
            icon=folium.Icon(color="purple", icon="tint", prefix="fa"),
        ).add_to(support_layer)

    # ---- stations snapped to the pixel graph (uncompressed only) ----
    for node in r["snapped_nodes"]:
        for s in Gu.nodes[node]["stations"]:
            folium.Marker(
                [float(s.get("latitude", coords(node)[0])), float(s.get("longitude", coords(node)[1]))],
                tooltip=f"WQ station {s.get('station_id')}",
                popup=popup_html(node, "WQ station (snapped to pixel)", {
                    "Station": s.get("station_id"),
                    "Name": s.get("station_name"),
                    "Type": s.get("station_type"),
                    "Snap distance": f"{s.get('distance_to_node_m', float('nan')):.0f} m",
                    "Years": f"{s.get('first_year')}–{s.get('last_year')} ({s.get('years_with_data')} with data)",
                    "Total results": s.get("total_results"),
                    **pixel_details(node),
                }),
                icon=folium.Icon(color="cadetblue", icon="vial", prefix="fa"),
            ).add_to(u_snapped)

    for view, entries in views.items():
        VIEW_LAYERS[f"{key}:{view}"] = entries


for r in rivers.values():
    draw_river(r)
    print("Drew", r["label"])


# ==================================================
# SEARCH INDEX (points from every river; picking one switches river)
# ==================================================

search_features = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(p.longitude), float(p.latitude)]},
        "properties": {
            "search_name": f"{rivers[p.river]['label']} | {p.type} | {p.label} | node {p.node_id}",
            "river": p.river,
        },
    }
    for p in points_df.itertuples()
]

search_layer = folium.GeoJson(
    {"type": "FeatureCollection", "features": search_features},
    name="Search index",
    marker=folium.CircleMarker(radius=1, opacity=0, fill_opacity=0),
    show=False,
    control=False,
).add_to(m)

plugins.Search(
    layer=search_layer, search_label="search_name",
    placeholder="Search station, node, source, junction...",
    collapsed=False, position="topleft", zoom=12,
).add_to(m)


# ==================================================
# MAP CONTROLS
# ==================================================

plugins.Fullscreen(position="topleft", title="Fullscreen", title_cancel="Exit fullscreen").add_to(m)
plugins.MiniMap(toggle_display=True, minimized=False).add_to(m)
plugins.MousePosition(position="bottomright", separator=" | ", prefix="Lat/Lon:", num_digits=5).add_to(m)
plugins.MeasureControl(position="topleft", primary_length_unit="kilometers",
                       secondary_length_unit="miles").add_to(m)


# ==================================================
# LEGEND (each row says which river:view it applies to; * = any)
# ==================================================

def legend_row(symbol, color, text, show="*:*"):
    return (f'<div class="rn-leg" data-show="{show}">'
            f'<span style="color:{color};">{symbol}</span> {text}</div>')

legend_html = (
    '<div id="rn-legend" style="position: fixed; bottom: 35px; left: 35px; z-index: 9999; '
    'background: rgba(255,255,255,0.94); border: 2px solid #777; border-radius: 8px; '
    'padding: 10px 12px; font-size: 13px; line-height: 1.45; '
    'box-shadow: 0 1px 6px rgba(0,0,0,0.25);">'
    '<b>River Network Legend</b> <span id="rn-legend-view" style="color:#555;"></span>'
    + legend_row("━", "#3388ff", "Compressed edge (follows river path)", "*:compressed")
    + legend_row("●", "#93c5fd", "Compressed node", "*:compressed")
    + legend_row("━", "#1e40af", "Pixel chain", "*:uncompressed")
    + legend_row("●", "#60a5fa", f"River-pixel node (zoom ≥ {PIXEL_MIN_ZOOM})", "*:uncompressed")
    + legend_row("●", "#facc15", "Pixel kept in compressed graph", "*:uncompressed")
    + legend_row("■", "#5f9ea0", "WQ station snapped to pixel",
                 " ".join(f"{k}:uncompressed" for k, r in rivers.items() if r["snapped_nodes"]))
    + legend_row("■", "#38aadd", "Water-quality station")
    + legend_row("■", "#7e22ce", "Hydrology support",
                 " ".join(f"{k}:*" for k, r in rivers.items() if r["support_nodes"]))
    + legend_row("▶", "#16a34a", "Source: ≤100 km to measurement")
    + legend_row("▶", "#f59e0b", "Source: &gt;100 km to measurement")
    + legend_row("▶", "#dc2626", "Source: no downstream measurement")
    + legend_row("■", "#15803d", "Outlet / downstream leaf")
    + legend_row("●", "#f59e0b", "Junction / meeting point")
    + "</div>"
)
m.get_root().html.add_child(Element(legend_html))

toggle_css = """
<style>
.rn-view { background: rgba(255,255,255,0.96); padding: 8px 10px; border-radius: 8px;
           box-shadow: 0 1px 6px rgba(0,0,0,0.3); font: 13px/1.4 system-ui, sans-serif; min-width: 250px; }
.rn-view .rn-label { font-weight: 600; margin: 2px 0 4px; }
.rn-view .rn-label + .rn-seg { margin-bottom: 8px; }
.rn-seg { display: flex; border: 1px solid #1e40af; border-radius: 6px; overflow: hidden; }
.rn-seg button { flex: 1; border: 0; background: #fff; color: #1e40af; padding: 5px 8px;
                 font: 600 13px system-ui, sans-serif; cursor: pointer; }
.rn-seg button + button { border-left: 1px solid #1e40af; }
.rn-seg button.active { background: #1e40af; color: #fff; }
.rn-stats { color: #333; }
.rn-note { margin-top: 4px; color: #6b7280; font-size: 12px; }
</style>
"""
m.get_root().header.add_child(Element(toggle_css))


# ==================================================
# PER-RIVER DATA FOR THE TOGGLE SCRIPT
# ==================================================

def pixel_rows(r):
    """[lat, lon, node, row, col, segment, dist_to_outlet_km, upstream_pixels, upstream_area_km2, elevation_m]"""
    rows = []
    for node, d in r["Gu"].nodes(data=True):
        upa, elev = d.get("upstream_area_km2"), d.get("elevation_m")
        rows.append([
            round(float(d["latitude"]), 5), round(float(d["longitude"]), 5), int(node),
            int(d["row"]), int(d["col"]), r["segment_of"].get(node, -1),
            round(r["dist_to_outlet_m"][node] / 1000, 1), r["upstream_count"][node],
            None if upa is None else round(float(upa), 1),
            None if elev is None else round(float(elev), 1),
        ])
    return rows


class JsBlock(MacroElement):
    """Raw JS rendered after every other map element (so all layer variables exist)."""
    _template = Template("{% macro script(this, kwargs) %}{{ this.code }}{% endmacro %}")

    def __init__(self, code):
        super().__init__()
        self._name = "JsBlock"
        self.code = code


def js_layers(entries):
    return "[" + ",".join(
        f"[{json.dumps(name)}, {layer.get_name()}, {str(show).lower()}]" for name, layer, show in entries
    ) + "]"


river_js = "{" + ",".join(
    f"""{json.dumps(key)}: {{
      label: {json.dumps(r['label'])},
      bounds: {json.dumps(r['bounds'])},
      P: {json.dumps(pixel_rows(r), separators=(",", ":"))},
      S: {json.dumps(r['segments'], separators=(",", ":"))},
      pixelLayer: {r['pixel_layer'].get_name()},
      stats: {json.dumps({
          "compressed": f"{r['Gc'].number_of_nodes():,} nodes · {r['Gc'].number_of_edges():,} edges",
          "uncompressed": f"{r['Gu'].number_of_nodes():,} river-pixel nodes · {r['Gu'].number_of_edges():,} edges",
      }, ensure_ascii=False)},
      views: {{ compressed: {js_layers(VIEW_LAYERS[key + ':compressed'])},
               uncompressed: {js_layers(VIEW_LAYERS[key + ':uncompressed'])} }}
    }}"""
    for key, r in rivers.items()
) + "}"

toggle_js = r"""
(function () {
  var map = __MAP__;
  var RIVERS = __RIVERS__;
  var MINZ = __MINZ__, MAXN = __MAXN__;
  var cur = { river: null, view: null };

  // ---------- layer control (basemaps + active river/view's overlays) ----------
  var lc = L.control.layers(
    { "OpenStreetMap": __T_OSM__, "CARTO Positron": __T_LIGHT__, "CARTO Dark Matter": __T_DARK__ },
    {}, { collapsed: false, position: "topright" }
  );

  // ---------- switches ----------
  var Toggle = L.Control.extend({
    options: { position: "topright" },
    onAdd: function () {
      var div = L.DomUtil.create("div", "rn-view");
      var riverButtons = Object.keys(RIVERS).map(function (k) {
        return '<button data-river="' + k + '">' + RIVERS[k].label + "</button>";
      }).join("");
      div.innerHTML =
        '<div class="rn-label">River</div><div class="rn-seg" id="rn-rivers">' + riverButtons + "</div>" +
        '<div class="rn-label">Network view</div>' +
        '<div class="rn-seg" id="rn-views"><button data-view="compressed">Compressed</button>' +
        '<button data-view="uncompressed">Uncompressed</button></div>' +
        '<div class="rn-stats" id="rn-stats"></div><div class="rn-note" id="rn-note"></div>';
      L.DomEvent.disableClickPropagation(div);
      L.DomEvent.disableScrollPropagation(div);
      div.querySelectorAll("button[data-river]").forEach(function (b) {
        b.addEventListener("click", function () { setState(b.getAttribute("data-river"), cur.view, true); });
      });
      div.querySelectorAll("button[data-view]").forEach(function (b) {
        b.addEventListener("click", function () { setState(cur.river, b.getAttribute("data-view"), false); });
      });
      return div;
    }
  });
  new Toggle().addTo(map);
  lc.addTo(map);

  function setState(river, view, fit) {
    if (river === cur.river && view === cur.view) return;
    Object.keys(RIVERS).forEach(function (k) {
      ["compressed", "uncompressed"].forEach(function (v) {
        RIVERS[k].views[v].forEach(function (e) {
          lc.removeLayer(e[1]);
          if (map.hasLayer(e[1])) map.removeLayer(e[1]);
        });
      });
      RIVERS[k].pixelLayer.clearLayers();
    });
    RIVERS[river].views[view].forEach(function (e) {
      lc.addOverlay(e[1], e[0]);
      if (e[2]) map.addLayer(e[1]);
    });
    var riverChanged = river !== cur.river;
    cur.river = river; cur.view = view;

    document.querySelectorAll("#rn-rivers button").forEach(function (b) {
      b.classList.toggle("active", b.getAttribute("data-river") === river);
    });
    document.querySelectorAll("#rn-views button").forEach(function (b) {
      b.classList.toggle("active", b.getAttribute("data-view") === view);
    });
    document.querySelectorAll(".rn-leg").forEach(function (row) {
      var ok = (row.getAttribute("data-show") || "").split(" ").some(function (t) {
        var p = t.split(":");
        return (p[0] === "*" || p[0] === river) && (p[1] === "*" || p[1] === view);
      });
      row.style.display = ok ? "" : "none";
    });
    document.getElementById("rn-stats").textContent = RIVERS[river].stats[view];
    var lv = document.getElementById("rn-legend-view");
    if (lv) lv.textContent = "(" + RIVERS[river].label + " · " + view + ")";
    if (history.replaceState) history.replaceState(null, "", "#" + river + "/" + view);
    if (fit && riverChanged) map.fitBounds(RIVERS[river].bounds);
    drawPixels();
  }

  // ---------- uncompressed pixel nodes: drawn only in view, when zoomed in ----------
  var renderer = L.canvas({ padding: 0.3 });

  function fmt(x) { return Number(x).toLocaleString(); }

  function pixelPopup(mk) {
    var R = RIVERS[mk._river], p = R.P[mk._i], s = p[5] >= 0 ? R.S[p[5]] : null;
    var g = "https://www.google.com/maps/search/?api=1&query=" + p[0] + "," + p[1];
    return "<b>River-pixel node</b> <span style='color:#666'>(" + R.label + ")</span>" +
      "<br><b>Graph node:</b> " + p[2] +
      "<br><b>Latitude:</b> " + p[0].toFixed(5) +
      "<br><b>Longitude:</b> " + p[1].toFixed(5) +
      "<br><b>Raster row / col:</b> " + p[3] + " / " + p[4] +
      "<br><b>Distance to outlet:</b> " + fmt(p[6]) + " km" +
      "<br><b>Upstream pixels:</b> " + fmt(p[7]) +
      (p[8] !== null ? "<br><b>Upstream area:</b> " + fmt(Math.round(p[8])) + " km²" : "") +
      (p[9] !== null ? "<br><b>Elevation:</b> " + p[9] + " m" : "") +
      (s ? "<br><b>Folded into compressed edge:</b> " + s[0] + " → " + s[1] +
           " (" + fmt(s[2]) + " pixel edges, " + s[3] + " km)"
         : "<br><b>Kept as a node in the compressed graph</b>") +
      '<br><a href="' + g + '" target="_blank">Open in Google Maps</a>';
  }

  function drawPixels() {
    var note = document.getElementById("rn-note");
    if (!note || !cur.river) return;
    var R = RIVERS[cur.river], layer = R.pixelLayer;
    layer.clearLayers();
    if (cur.view !== "uncompressed" || !map.hasLayer(layer)) { note.textContent = ""; return; }
    var z = map.getZoom();
    if (z < MINZ) {
      note.textContent = "Zoom in to level " + MINZ + "+ to see individual river-pixel nodes (now " + z + ").";
      return;
    }
    var b = map.getBounds().pad(0.15), n = 0, rad = z >= 14 ? 4.5 : (z >= 13 ? 3.5 : 2.5);
    for (var i = 0; i < R.P.length; i++) {
      var p = R.P[i];
      if (!b.contains([p[0], p[1]])) continue;
      if (++n > MAXN) break;
      var mk = L.circleMarker([p[0], p[1]], {
        renderer: renderer, radius: rad, color: "#1e3a8a", weight: 1,
        fillColor: p[5] < 0 ? "#facc15" : "#60a5fa", fillOpacity: 0.9
      });
      mk._i = i; mk._river = cur.river;
      mk.bindPopup(pixelPopup, { maxWidth: 440 });
      mk.bindTooltip("Pixel node " + p[2]);
      layer.addLayer(mk);
    }
    note.textContent = n > MAXN
      ? "Showing " + fmt(MAXN) + " river-pixel nodes in view — zoom in for all."
      : fmt(n) + " river-pixel nodes in view.";
  }

  map.on("moveend", drawPixels);
  map.on("overlayadd overlayremove", function (e) {
    if (cur.river && e.layer === RIVERS[cur.river].pixelLayer) drawPixels();
  });

  // Picking a search result from another river switches to that river
  __SEARCH__.on("search:locationfound", function (e) {
    var props = e.layer && e.layer.feature && e.layer.feature.properties;
    if (props && RIVERS[props.river] && props.river !== cur.river) setState(props.river, cur.view, false);
  });

  // Start from the URL hash: #river/view, #river, or #view (= default river)
  var parts = (location.hash || "").replace("#", "").split("/");
  var river = RIVERS[parts[0]] ? parts[0] : __DEFAULT__;
  var view = ["compressed", "uncompressed"].indexOf(parts[parts.length - 1]) >= 0 ? parts[parts.length - 1] : "compressed";
  setState(river, view, true);
})();
"""

replacements = {
    "__MAP__": m.get_name(),
    "__RIVERS__": river_js,
    "__MINZ__": str(PIXEL_MIN_ZOOM),
    "__MAXN__": str(PIXEL_MAX_IN_VIEW),
    "__T_OSM__": tile_osm.get_name(),
    "__T_LIGHT__": tile_light.get_name(),
    "__T_DARK__": tile_dark.get_name(),
    "__SEARCH__": search_layer.get_name() + "searchControl",   # variable name folium's Search plugin uses
    "__DEFAULT__": json.dumps(DEFAULT_RIVER),
}
for key, value in replacements.items():
    toggle_js = toggle_js.replace(key, value)

m.add_child(JsBlock(toggle_js))
m.fit_bounds(start_bounds)


# ==================================================
# SAVE
# ==================================================

m.save(OUTPUT_HTML)
print("Saved interactive map:", OUTPUT_HTML, f"({os.path.getsize(OUTPUT_HTML) / 1e6:.1f} MB)")

if SITE_HTML:
    m.save(SITE_HTML)
    print("Updated site:", SITE_HTML)

# The page is several MB, so it is not rendered inline here.
# Open the HTML file in a browser (or run the next cell to embed it).

In [ ]:
# Optional: preview inside the notebook (adds several MB to the .ipynb if you save it)
# from IPython.display import IFrame
# IFrame(OUTPUT_HTML, width="100%", height=700)

## 5. Optional quick summaries

In [ ]:
source_coverage = pd.concat([r["source_coverage"] for r in rivers.values()], ignore_index=True)
display(source_coverage.sort_values(["river", "distance_to_first_measurement_km"],
                                    ascending=[True, False], na_position="first"))

In [ ]:
display(points_df.sort_values(["river", "type", "label"]))